# Dog Bounding Box Detection
Uses a pre-trained YOLOv5 model to detect dogs and draw bounding boxes.

In [1]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2
import numpy as np
from audio_utils import load_data, spectrogram, plot_spectrogram
from scipy.optimize import linear_sum_assignment
import torch
from ultralytics import YOLO

In [2]:
# Load YOLOv8 model
model = YOLO('yolov8n.pt')

# Load font
font = ImageFont.truetype('/Library/Fonts/Arial.ttf', 40)

In [3]:
class StableTracker:
    def __init__(self, iou_threshold=0.3, max_age=5):
        self.next_stable_id = 0
        self.active = {} # {'box': (x1, y1, x2, y2), 'last_seen': frame_id, 'yolo_id': yolo_id}
        self.lost = {}
        self.yolo_to_stable = {}
        self.iou_threshold = iou_threshold
        self.max_age = max_age
        self.colors = {}

    def _iou(self, a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        ix1, ix2 = max(ax1, bx1), min(ax2, bx2)
        iy1, iy2 = max(ay1, by1), min(ay2, by2)
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)

        intersection = (iw * ih) if iw > 0 and ih > 0 else 0
        union = ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - intersection)

        return intersection / union if union > 0 else 0
    
    def update(self, detections, frame_id):
        unmatched_detections = []
        for box, yolo_id in detections:
            if yolo_id in self.yolo_to_stable:
                stable_id = self.yolo_to_stable[yolo_id]

                if stable_id in self.lost:
                    self.active[stable_id] = self.lost.pop(stable_id)
                if stable_id in self.active:
                    self.active[stable_id].update({'box': box, 'last_seen': frame_id, 'yolo_id': yolo_id})
            else:
                unmatched_detections.append((box, yolo_id))

        # Match unmatched detections to lost trackers
        if unmatched_detections and self.lost:
            lost_ids = list(self.lost.keys())
            lost_boxes = [self.lost[stable_id]['box'] for stable_id in lost_ids]

            cost_matrix = np.zeros((len(unmatched_detections), len(lost_ids)), dtype=np.float32)
            for i, (box, _) in enumerate(unmatched_detections):
                for j, lost_box in enumerate(lost_boxes):
                    cost_matrix[i, j] = 1 - self._iou(box, lost_box)

            opt_row, opt_col = linear_sum_assignment(cost_matrix) # Hungarian algorithm for optimal assignment
            still_unmatched = []

            for i, (box, yolo_id) in enumerate(unmatched_detections):
                matched = False
                for r, c in zip(opt_row, opt_col):
                    if r == i and cost_matrix[r, c] < (1 - self.iou_threshold):
                        stable_id = lost_ids[c]
                        self.active[stable_id] = {'box': box, 'last_seen': frame_id, 'yolo_id': yolo_id}
                        self.yolo_to_stable[yolo_id] = stable_id
                        del self.lost[stable_id]
                        matched = True
                        break
                if not matched:
                    still_unmatched.append((box, yolo_id))
            
            unmatched_detections = still_unmatched

        # Add remaining unmatched detections as new trackers
        for box, yolo_id in unmatched_detections:
            stable_id = self.next_stable_id
            self.next_stable_id += 1
            self.yolo_to_stable[yolo_id] = stable_id
            self.active[stable_id] = {'box': box, 'last_seen': frame_id, 'yolo_id': yolo_id}
        
        to_remove = []

        for stable_id, data in self.active.items():
            if frame_id - data['last_seen'] > self.max_age:
                to_remove.append(stable_id)

        for stable_id in to_remove:
            self.lost[stable_id] = self.active.pop(stable_id)

        final_active = []
        for stable_id, data in self.active.items():
            final_active.append((data['box'], stable_id))

        return final_active



In [ ]:
# Load dog image
img = Image.open('files/wynnie.jpg').convert('RGB')

# Run inference
results = model(img, device='cpu')


# Draw bounding boxes for dogs
draw = ImageDraw.Draw(img)
dog_count = 0

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])

        if label == 'dog' and conf > 0.5:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            draw.rectangle([x1, y1, x2, y2], outline='blue', width=4)
            draw.rectangle([x1, y1 - 45, x1 + 200, y1], fill='blue')
            draw.text((x1 + 4, y1 - 40), f'{label} {conf:.2f}', fill='white', font=font)

# Display result
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Make dog bark spectrogram
vid = 'files/dogs1.mp4'
s, framerate = load_data(vid)

freqs, times, power, log_spec = spectrogram(vid, s, framerate)
plot_spectrogram(freqs, times, log_spec)

In [ ]:
# Helper to get spectrogram column for time t
def get_spec_col(t):
    return np.argmin(np.abs(times - t))

stable_tracker = StableTracker()

# Load dog video
cap = cv2.VideoCapture('files/dogs1.mp4')

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Precompute spectrogram visualization
spec_full = np.flipud(log_spec)

spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
spec_vis_full = spec_vis_full.astype(np.uint8)
spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
spec_vis_full = cv2.resize(spec_vis_full, (w, h))

# Video writer for output

out = cv2.VideoWriter(
    'files/dogs1-output.mp4',
    cv2.VideoWriter_fourcc(*'avc1'),
    fps,
    (w * 2, h)
)

frame_id = 0
interval = 3  # Run detection every 3 frames

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps # Current time in seconds
    spec_id = get_spec_col(t)

    spec_vis = spec_vis_full.copy()

    x_pos = int(spec_id / log_spec.shape[1] * w)

    cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

    # Run tracking
    if frame_id % interval == 0:
        results = model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            imgsz=640,
            conf=0.25,
            verbose=False
        )

    r = results[0]

    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        yolo_ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

        detections = []
        for i in range(len(boxes)):
            if model.names[int(cls_ids[i])] == 'dog' and confs[i] > 0.25:
                detections.append((boxes[i], yolo_ids[i]))

        stable_results = stable_tracker.update(detections, frame_id)

        for box, stable_id in stable_results:
            x1, y1, x2, y2 = map(int, box)

            cv2.rectangle(frame, (x1, y1 - 45), (x1 + 100, y1), (255, 0, 0), -1)
            cv2.putText(frame, f'dog #{stable_id}',
                        (x1 + 5, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (255, 255, 255), 2)

    frame = frame.astype('uint8')
    frame = np.ascontiguousarray(frame)

    frame = cv2.resize(frame, (w, h))
    spec_vis = cv2.resize(spec_vis, (w, h))
    frame = np.hstack([frame, spec_vis])

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
cv2.destroyAllWindows()

In [ ]:
# Helper to get spectrogram column for time t
def get_spec_col(t):
    return np.argmin(np.abs(times - t))

# Load dog video
cap = cv2.VideoCapture('files/dogs1.mp4')

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Precompute spectrogram visualization
spec_full = np.flipud(log_spec)

spec_vis_full = cv2.normalize(spec_full, None, 0, 255, cv2.NORM_MINMAX)
spec_vis_full = spec_vis_full.astype(np.uint8)
spec_vis_full = cv2.applyColorMap(spec_vis_full, cv2.COLORMAP_INFERNO)
spec_vis_full = cv2.resize(spec_vis_full, (w, h))

# Video writer for output

out = cv2.VideoWriter(
    'files/dogs1-output.mp4',
    cv2.VideoWriter_fourcc(*'avc1'),
    fps,
    (w * 2, h)
)

frame_id = 0
interval = 3  # Run detection every 3 frames

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps # Current time in seconds
    spec_id = get_spec_col(t)

    spec_vis = spec_vis_full.copy()

    x_pos = int(spec_id / log_spec.shape[1] * w)

    cv2.line(spec_vis, (x_pos, 0), (x_pos, h), (0, 255, 255), 3)

    # Run tracking
    if frame_id % interval == 0:
        results = model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            imgsz=640,
            conf=0.25,
            verbose=False
        )

    r = results[0]

    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None

        for i, box in enumerate(boxes):
            label = model.names[int(cls_ids[i])]
            conf = confs[i]

            if label == 'dog' and confs[i] > 0.5:
                x1, y1, x2, y2 = map(int, box)

                track_id = int(ids[i]) if ids is not None else -1

                # Draw box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)

                text = f'{label} #{track_id} {conf:.2f}'

                cv2.rectangle(frame, (x1, y1 - 45), (x1 + 100, y1), (255, 0, 0), -1)
                cv2.putText(frame, text,
                            (x1 + 5, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255, 255, 255), 2)

    frame = frame.astype('uint8')
    frame = np.ascontiguousarray(frame)

    frame = cv2.resize(frame, (w, h))
    spec_vis = cv2.resize(spec_vis, (w, h))
    frame = np.hstack([frame, spec_vis])

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
cv2.destroyAllWindows()